In [ ]:
import os, json
import numpy as np
import scipy.io
from PIL import Image

def as_scalar(x):
    x = np.array(x)
    return float(x.squeeze())

def load_pdata(mat):
    pdata = mat["PData"][0,0]
    PDelta = np.array(pdata["PDelta"]).squeeze()   # [dx, dy, dz]
    Origin = np.array(pdata["Origin"]).squeeze()   # [ox, oy, oz]
    Size   = np.array(pdata["Size"]).squeeze()     # [H, W, 1]
    return PDelta, Origin, Size

def iq_to_uint8(frame):
    # frame: complex 2D
    mag = np.abs(frame).astype(np.float32)

    # log compression (évite la dynamique énorme)
    mag = np.log1p(mag)

    # normalize per-frame -> 0..255
    mmin, mmax = float(mag.min()), float(mag.max())
    if mmax <= mmin + 1e-9:
        return np.zeros_like(mag, dtype=np.uint8)
    img = (mag - mmin) / (mmax - mmin)
    img = (img * 255.0).clip(0, 255).astype(np.uint8)
    return img

def points_to_bboxes(ListPos_frame, PDelta, Origin, H, W, box_size_px=6):
    # ListPos_frame: (N,4)
    # On utilise x_mm = col0, z_mm = col2 (comme observé sur ton fichier)
    dx = as_scalar(PDelta[0])
    dz = as_scalar(PDelta[2])
    ox = as_scalar(Origin[0])
    oz = as_scalar(Origin[2])

    bboxes = []
    for row in ListPos_frame:
        x_mm, y_mm, z_mm, flag = row
        if np.any(np.isnan([x_mm, z_mm])): 
            continue
        # certains points ont flag=0/1 (dans ton fichier ça existe)
        if not np.isnan(flag) and float(flag) <= 0:
            continue

        # mm -> pixel
        x_px = int(round((float(x_mm) - ox) / dx))
        z_px = int(round((float(z_mm) - oz) / dz))

        # bbox centrée
        half = box_size_px // 2
        x0 = max(0, x_px - half)
        y0 = max(0, z_px - half)
        x1 = min(W - 1, x_px + half)
        y1 = min(H - 1, z_px + half)

        w = max(1, x1 - x0 + 1)
        h = max(1, y1 - y0 + 1)

        # Filtre si hors image
        if x0 >= W or y0 >= H or x1 < 0 or y1 < 0:
            continue

        bboxes.append((x0, y0, w, h))
    return bboxes

def write_coco(split_dir, images_info, annotations, categories):
    out_path = os.path.join(split_dir, "_annotations.coco.json")
    coco = {
        "images": images_info,
        "annotations": annotations,
        "categories": categories
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False)

def convert_mat_to_coco(mat_path, out_root, train_ratio=0.8, box_size_px=6):
    os.makedirs(out_root, exist_ok=True)
    train_dir = os.path.join(out_root, "train")
    valid_dir = os.path.join(out_root, "valid")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)

    mat = scipy.io.loadmat(mat_path)
    IQ = mat["IQ"]  # (H,W,T) complex
    ListPos = mat["ListPos"]  # (N,4,T)
    PDelta, Origin, Size = load_pdata(mat)

    H, W, T = IQ.shape
    base = os.path.splitext(os.path.basename(mat_path))[0]

    categories = [{"id": 1, "name": "particle"}]

    images_train, ann_train = [], []
    images_valid, ann_valid = [], []

    ann_id = 1
    img_id = 1

    split_T = int(round(T * train_ratio))

    for t in range(T):
        split_dir = train_dir if t < split_T else valid_dir

        img_u8 = iq_to_uint8(IQ[:, :, t])
        filename = f"{base}_frame{t:04d}.png"
        Image.fromarray(img_u8).save(os.path.join(split_dir, filename))

        img_info = {"id": img_id, "file_name": filename, "width": W, "height": H}
        bboxes = points_to_bboxes(ListPos[:, :, t], PDelta, Origin, H, W, box_size_px=box_size_px)

        # ajoute l'image même si 0 bbox (tu peux changer si tu veux)
        if split_dir == train_dir:
            images_train.append(img_info)
        else:
            images_valid.append(img_info)

        for (x, y, w, h) in bboxes:
            ann = {
                "id": ann_id,
                "image_id": img_id,
                "category_id": 1,
                "bbox": [float(x), float(y), float(w), float(h)],
                "area": float(w * h),
                "iscrowd": 0
            }
            if split_dir == train_dir:
                ann_train.append(ann)
            else:
                ann_valid.append(ann)
            ann_id += 1

        img_id += 1

    write_coco(train_dir, images_train, ann_train, categories)
    write_coco(valid_dir, images_valid, ann_valid, categories)

    print("OK ->", out_root)
    print("train images:", len(images_train), "train anns:", len(ann_train))
    print("valid images:", len(images_valid), "valid anns:", len(ann_valid))

if __name__ == "__main__":
    convert_mat_to_coco(
        mat_path=r"C:\Users\thoma\dev_projet\ecptr\PALA\PALA\PALA_data\PALA_data_InSilicoFlow\IQ",
        out_root=r"C:/Users/thoma/dev_projet/ecptr/dataset_coco",
        train_ratio=0.8,
        box_size_px=6
    )
